# SQL 14: interview concepts

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-14-interview-concepts.ipynb)

Interviewers rarely ask for a single query. They ask why a query behaves the way it does, what an index is for,
and which trade-off you would choose. This chapter answers the core concept questions in short, direct form. Each
answer says what to remember in the interview, and includes a runnable demo where Spark can show it.

The concepts are grouped into four areas: how queries are evaluated, data modeling, storage and performance, and
correctness traps. Practice questions with answers follow the list.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

The setup creates eight customers in four regions and 14 orders: 11 completed, 2 cancelled and 1 pending. Customers 7
and 8 have no orders. Some demos below need data stored as files, because plans for files show what the engine
actually reads. The next cell writes the orders as Parquet files, split into one directory per month, and exposes
them as the view `orders_files`.

In [ ]:
import math
import re
import tempfile

files_dir = tempfile.mkdtemp()
spark.sql("SELECT *, date_format(order_date, 'yyyy-MM') AS order_month FROM orders").write.partitionBy(
    "order_month"
).parquet(files_dir + "/orders")
spark.read.parquet(files_dir + "/orders").createOrReplaceTempView("orders_files")


def plan(query):
    """Return the physical plan Spark chose for a query, as text."""
    return spark.sql("EXPLAIN " + query).first()[0]


assert spark.sql("SELECT COUNT(*) FROM orders_files").collect() == [(14,)]

## 1. How queries are evaluated

SQL says *what* result you want, not *how* to compute it. The engine's **optimiser** turns the query into a
**plan**: a tree of steps such as scan, filter, join and aggregate. The questions in this section are about that
translation.

### Q: What does `EXPLAIN` show, and what should you look for?

**Answer.** The plan the engine chose: which tables it scans, which filters it **pushes down** (applies while
reading, so fewer rows leave the scan), which join strategy it uses, and where it moves data between machines.

Look for full scans of large tables, `Exchange` nodes, and joins that are slower than their input sizes justify. An
`Exchange` is a **shuffle**: rows are sent across the network so that rows with the same key end up on the same
machine. `EXPLAIN FORMATTED` prints the plan with details. Engines differ on `EXPLAIN ANALYZE`, which runs the query
and reports actual row counts and timings.

The plan below joins the order files to customers and sums by region. Reading from the bottom up, it shows:

- `PushedFilters: [..., EqualTo(status,completed)]`: the status filter is applied inside the file scan.
- `BroadcastHashJoin`: the small side is copied to every machine, so the large side is not shuffled.
- `Exchange hashpartitioning(region, ...)`: one shuffle, to bring each region's partial sums together.

In [ ]:
p = plan("""
SELECT c.region, SUM(o.amount) AS total
FROM orders_files o
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.status = 'completed'
GROUP BY c.region
""")
print(p)
assert "EqualTo(status,completed)" in p
assert "BroadcastHashJoin" in p
assert "Exchange hashpartitioning(region" in p

### Where you'll use it

- **Slow dashboards:** the plan shows whether a filter runs before the join and whether a join scans a whole table. It is the first thing to read when a report times out.
- **Reviewing a change:** comparing the plan before and after a rewrite shows whether a shuffle or a scan disappeared.

### Q: What is the difference between `WHERE` and `HAVING`, and between `GROUP BY` and a window function?

**Answer.** A query runs its clauses in a logical order: `FROM` and joins, then `WHERE`, then `GROUP BY`, then
`HAVING`, then `SELECT`, then `ORDER BY`. So `WHERE` filters rows before grouping, and `HAVING` filters groups after
aggregation. Moving a condition between them changes the answer.

Worked by hand, the completed revenue per customer is 450, 700, 300, 320, 950 and 275 for customers 1 to 6. With
`WHERE status = 'completed'` and `HAVING SUM(amount) > 500`, customers 2 (700) and 5 (950) remain. Without the
`WHERE`, cancelled and pending orders count too: customer 4 rises to 540 and customer 6 to 600, so four customers pass.

In [ ]:
q = """
SELECT customer_id, SUM(amount) AS total
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
HAVING SUM(amount) > 500
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, 700), (5, 950)]

q_all = "SELECT customer_id FROM orders GROUP BY customer_id HAVING SUM(amount) > 500 ORDER BY customer_id"
assert spark.sql(q_all).collect() == [(2,), (4,), (5,), (6,)]

`GROUP BY` returns one row per group. A window function keeps every input row and adds a computed column. Over the
14 orders, grouping by customer returns 6 rows, while `SUM(amount) OVER (PARTITION BY customer_id)` returns all 14,
each carrying its customer's total.

In [ ]:
grouped = spark.sql("SELECT customer_id, SUM(amount) FROM orders GROUP BY customer_id").count()
windowed = spark.sql("SELECT order_id, SUM(amount) OVER (PARTITION BY customer_id) FROM orders").count()
assert (grouped, windowed) == (6, 14)

### Where you'll use it

- **Report logic:** `WHERE` removes rows before grouping (for example, only 2024 orders); `HAVING` filters the groups afterwards (for example, customers above 1,000 in spend).
- **Row-level context:** a window adds "customer total" next to every order, which a `GROUP BY` cannot do without a join back.

### Q: `CTE` or subquery? Does a CTE make a query faster?

**Answer.** A **CTE** (common table expression, the `WITH name AS (...)` form) is for readability: it names a step so
the query reads top to bottom. It does not guarantee speed. Some engines compute a CTE once and store it; others
**inline** it, copying its text into each place it is used, so it runs once per reference. Check the plan before you
assume either. The aggregations chapter shows CTEs, subqueries and a recursive CTE.

The two queries below are the same computation written both ways, and return the same rows.

In [ ]:
with_cte = """
WITH completed AS (SELECT customer_id, amount FROM orders WHERE status = 'completed')
SELECT customer_id, SUM(amount) AS total FROM completed GROUP BY customer_id ORDER BY customer_id
"""
with_subquery = """
SELECT customer_id, SUM(amount) AS total
FROM (SELECT customer_id, amount FROM orders WHERE status = 'completed') AS completed
GROUP BY customer_id ORDER BY customer_id
"""
assert spark.sql(with_cte).collect() == spark.sql(with_subquery).collect()
assert len(spark.sql(with_cte).collect()) == 6

### Where you'll use it

- **Code review:** a dbt model written as chained CTEs is easier to test step by step than one deep nested subquery, even when the engine produces the same plan.

## 2. Data modeling

Data modeling decides which tables exist, what one row of each table means, and how tables refer to each other.
One row's meaning is the table's **grain**, for example "one row per order".

### Q: What is normalisation? Explain 1NF, 2NF and 3NF, and when would you denormalise?

**Answer.** **Normalisation** stores each fact once, so a change is made in one place. The normal forms are rules
that remove repeated facts step by step:

| Form | Rule | Typical violation |
| --- | --- | --- |
| 1NF | every column holds one atomic value; no repeating groups | a `phones` column holding `"555-1, 555-2"` |
| 2NF | every non-key column depends on the whole key | in `(order_id, product_id)`, a `product_name` that depends only on `product_id` |
| 3NF | no non-key column depends on another non-key column | `customer_city` stored on orders, although it depends on `customer_id` |

Normalise transactional systems to avoid **update anomalies**: a fact stored in several rows can be changed in some
rows and not others. Denormalise analytics tables, such as a fact table that carries the customer's region, when reads
matter more than writes. Keep the grain clear either way.

The demo shows both costs of a denormalised table. First, the customer's name is repeated on every order, so 14 rows
describe 6 customers and counting customers needs `DISTINCT`. Second, renaming customer 1 must touch all three of its
rows. If one row is missed, the customer has two names, which is the update anomaly.

In [ ]:
q = """
WITH denormalised AS (
  SELECT o.order_id, c.customer_id, c.name AS customer_name
  FROM orders o JOIN customers c ON c.customer_id = o.customer_id
)
SELECT COUNT(*) AS rows_, COUNT(DISTINCT customer_id) AS customers FROM denormalised
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(14, 6)]

q = """
WITH denormalised AS (
  SELECT o.order_id, c.customer_id, c.name AS customer_name
  FROM orders o JOIN customers c ON c.customer_id = o.customer_id
),
partly_renamed AS (
  SELECT order_id, customer_id,
         CASE WHEN order_id IN (1, 2) THEN 'Acme Group' ELSE customer_name END AS customer_name
  FROM denormalised
)
SELECT customer_id, COUNT(DISTINCT customer_name) AS names
FROM partly_renamed WHERE customer_id = 1 GROUP BY customer_id
"""
assert spark.sql(q).collect() == [(1, 2)]

### Where you'll use it

- **Operational systems:** an order system stores each customer once and refers to them by key, so a change of address is one update.
- **Analytics:** a warehouse often denormalises on purpose, so a dashboard does not repeat joins for every query.

### Q: Star schema or snowflake schema?

**Answer.** A **fact table** records events at a fixed grain (one row per order) with numeric measures and keys. A
**dimension table** describes the things those keys point to (customers, products, dates). A **star schema** joins
the fact table directly to denormalised dimension tables. A **snowflake schema** splits each dimension into
normalised sub-tables, such as `product -> brand -> category`.

```sql
-- star: one join reaches the category
SELECT p.category, SUM(f.amount) FROM fact_sales f JOIN dim_product p ON p.product_id = f.product_id GROUP BY 1;

-- snowflake: the category is two joins further away
SELECT c.category_name, SUM(f.amount)
FROM fact_sales f
JOIN dim_product p  ON p.product_id = f.product_id
JOIN dim_brand b    ON b.brand_id = p.brand_id
JOIN dim_category c ON c.category_id = b.category_id
GROUP BY 1;
```

Stars are simpler to query and usually faster for analytics; snowflakes save storage and keep dimension values
consistent. The modeling chapter builds a star schema.

### Where you'll use it

- **BI tools:** Looker, Power BI and Tableau models are built as star schemas, because the flat dimensions give simpler, faster queries.

### Q: What is a slowly changing dimension? Type 1, 2 and 3?

**Answer.** A **slowly changing dimension** (SCD) is a dimension whose attributes change now and then, such as a
customer's city. The type says how the change is stored.

- **Type 1** overwrites the old value. History is lost.
- **Type 2** adds a row per version, with validity dates. History is kept.
- **Type 3** adds a column for the previous value. Only one change of history is kept.

A Type 2 row is valid on a **half-open interval**: from `valid_from` (included) up to `valid_to` (excluded). On the
change date exactly one version matches, because the old row's `valid_to` equals the new row's `valid_from`.

Worked by hand: customer 1 lived in Oslo until 2023-06-01, then Lima. An order on 2023-03-10 falls in the first
interval and joins to Oslo. Orders on 2023-06-01 and 2023-07-01 fall in the second and join to Lima.

In [ ]:
q = """
WITH dim_customer AS (
  SELECT * FROM VALUES
    (1, 'Oslo', DATE '2022-01-01', DATE '2023-06-01'),
    (1, 'Lima', DATE '2023-06-01', DATE '9999-12-31')
  AS d(customer_id, city, valid_from, valid_to)
),
sales AS (
  SELECT * FROM VALUES (1, DATE '2023-03-10'), (1, DATE '2023-06-01'), (1, DATE '2023-07-01')
  AS s(customer_id, sale_date)
)
SELECT s.sale_date, d.city
FROM sales s
JOIN dim_customer d
  ON d.customer_id = s.customer_id AND s.sale_date >= d.valid_from AND s.sale_date < d.valid_to
ORDER BY s.sale_date
"""
spark.sql(q).show()
assert [r.city for r in spark.sql(q).collect()] == ["Oslo", "Lima", "Lima"]

### Where you'll use it

- **History reports:** a customer moves from Oslo to Lima. A Type 2 dimension keeps both rows, so last year's revenue still shows Oslo.

## 3. Storage and performance

Most query time goes into reading data and moving it between machines. The questions here are about structures
that let an engine read less: indexes, partitions and columnar files.

### Q: What is an index? What does a B-tree index do?

**Answer.** An **index** is a separate structure, kept in sync with the table, that maps a key to the rows holding it.
A **B-tree** is a balanced tree whose nodes each hold hundreds of sorted keys. A lookup walks from the root to a
leaf, choosing one child per level, so the number of steps grows with the logarithm of the row count instead of the
row count itself.

Worked numbers: with 500 keys per node, a billion rows need four levels, because 500 to the power 3 is 125 million
(too few) and 500 to the power 4 is 62.5 billion (enough). A full scan reads all billion rows.

Indexes speed up **point lookups** (one key) and selective range filters. They add cost to every write, because the
index must be updated too, and they use storage. Spark has no user-created indexes; it relies on partitioning, file
statistics and data skipping. Databases such as PostgreSQL, MySQL and SQL Server expose indexes directly.

In [ ]:
rows_in_table = 10**9
keys_per_node = 500
assert keys_per_node**3 < rows_in_table <= keys_per_node**4
assert math.ceil(math.log(rows_in_table, keys_per_node)) == 4

### Where you'll use it

- **Point lookups:** a B-tree index on `email` finds one customer in a large `customers` table without scanning it. OLTP databases such as PostgreSQL and MySQL rely on this.

### Q: Clustered vs non-clustered index. What is a covering index?

**Answer.** A **clustered** index stores the table's rows themselves in index order, so a table can have only one. A
**non-clustered** index is a separate structure whose entries point to the rows, so finding a row costs one more
read. A **covering** index contains every column a query needs, so the engine answers from the index and never reads
the table.

```sql
-- PostgreSQL: the index covers queries that filter on customer_id and read only order_date and amount
CREATE INDEX idx_orders_customer ON orders (customer_id) INCLUDE (order_date, amount);
SELECT order_date, amount FROM orders WHERE customer_id = 42;   -- index-only scan
```

### Where you'll use it

- **Covering indexes:** an index that already holds every column a dashboard query needs, so the table itself is never read.

### Q: What is partitioning and how does it help?

**Answer.** **Partitioning** splits a table into pieces by the value of a column, such as one directory per month. A
query that filters on the partition column reads only the matching pieces, which is called **partition pruning**.

`orders_files` is partitioned by `order_month`. In the plan, `PartitionFilters` lists the conditions used to skip
directories. Filtering on March puts `order_month = 2023-03` there, so only one of the three directories is read.

In [ ]:
p = plan("SELECT amount FROM orders_files WHERE order_month = '2023-03'")
assert "(order_month#" in p and "= 2023-03)" in p.split("PartitionFilters:")[1].split("]")[0]
assert spark.sql("SELECT COUNT(*) FROM orders_files WHERE order_month = '2023-03'").collect() == [(5,)]

The risks are too many small partitions, which means many small files and slow listing, and filters that the engine
cannot match to a partition. Spark can prune on any condition that uses only partition columns, even inside a
function such as `substr(order_month, 1, 4) = '2023'`. A filter on a different column, such as `order_date`, cannot
prune `order_month` directories, even though the two are related. The `INSERT OVERWRITE` statement in the views
chapter replaces partitions in the same directory layout.

In [ ]:
def partition_filters(query):
    return re.search(r"PartitionFilters: \[([^\]]*)\]", plan(query)).group(1)


assert "substr(order_month" in partition_filters("SELECT amount FROM orders_files WHERE substr(order_month, 1, 4) = '2023'")
assert partition_filters("SELECT amount FROM orders_files WHERE order_date >= DATE '2023-03-01'") == ""

### Where you'll use it

- **Time-based tables:** a fact table split by month lets a query for one month read one partition, and lets old partitions be dropped cheaply.

### Q: How would you approach optimising a slow query?

**Answer.** Measure first. Then, in order:

1. Check the plan for full scans, large shuffles and the wrong join strategy.
2. Make filters **sargable** (see section 5): compare bare columns with constants, and keep types consistent.
3. Reduce the data early: filter before joining, select only the columns needed, and avoid `SELECT *`.
4. Fix the join: broadcast a small table, and check for row multiplication and skew.
5. Change the layout last: partition or cluster on the column your queries filter by.

Re-run and compare the plan and the timing after each change.

### Where you'll use it

- **Triage in production:** check the plan and the row counts, fix the join or filter that multiplies data, then consider an index or partition.
- **Index and partition decisions:** OLTP databases (PostgreSQL, MySQL) use indexes; warehouses use partitioning and clustering. The same question, two toolkits.

## 4. Correctness and transactions

These questions are about getting the right answer: what concurrent users see, what a statement removes, and the
places where `NULL` and joins silently change a result.

### Q: What does ACID mean, and what are the isolation levels?

**Answer.** A **transaction** is a group of statements that the database treats as one unit. ACID names four
guarantees about transactions:

- **Atomicity:** a transaction takes effect completely or not at all.
- **Consistency:** it moves the database from one valid state to another, so constraints hold before and after.
- **Isolation:** concurrent transactions do not interfere in ways the isolation level forbids.
- **Durability:** committed changes survive a crash.

Isolation levels are defined by which **anomalies** they allow. Each anomaly is something transaction A can see
while transaction B is running:

- **Dirty read:** A reads a row B has changed but not committed. If B rolls back, A has seen data that never existed.
- **Non-repeatable read:** A reads a row twice and gets different values, because B updated and committed in between.
- **Phantom read:** A runs the same filter twice and gets a different set of rows, because B inserted or deleted a
  matching row in between.

| Level | Dirty read | Non-repeatable read | Phantom read |
| --- | --- | --- | --- |
| Read uncommitted | possible | possible | possible |
| Read committed | no | possible | possible |
| Repeatable read | no | no | possible |
| Serializable | no | no | no |

Stronger levels allow fewer anomalies and cost more locking or more retries. Warehouses and table formats such as
Delta and Iceberg usually give **snapshot isolation**: each query reads a consistent snapshot of the table, and writes
commit atomically. Spark's built-in tables and temporary views do not.

### Where you'll use it

- **Money movement:** a bank transfer must debit one account and credit another as one unit. ACID transactions guarantee that a failure leaves neither change applied.
- **Reports during loads:** the isolation level decides whether a report run during a load sees half of the new rows.

### Q: What is the difference between `DELETE`, `TRUNCATE` and `DROP`?

**Answer.** `DELETE` removes the rows that match a condition, row by row, and is logged, so it can be rolled back in a
transaction. `TRUNCATE` removes all rows at once by releasing the table's storage, which is much faster. Whether it
can be rolled back depends on the engine: PostgreSQL and SQL Server allow it inside a transaction, MySQL and Oracle
commit it immediately. `DROP` removes the table itself, including its definition. Table formats may log `DELETE` as a
new commit, which keeps the old rows available for time travel.

Spark's Parquet tables support `TRUNCATE` and `DROP` (not `DELETE`, as the views chapter shows). After `TRUNCATE` the
table exists with no rows; after `DROP` it no longer exists.

In [ ]:
spark.sql("DROP TABLE IF EXISTS truncate_demo")
spark.sql("CREATE TABLE truncate_demo USING PARQUET AS SELECT * FROM orders")
spark.sql("TRUNCATE TABLE truncate_demo")
assert spark.catalog.tableExists("truncate_demo")
assert spark.sql("SELECT COUNT(*) FROM truncate_demo").collect() == [(0,)]

spark.sql("DROP TABLE truncate_demo")
assert not spark.catalog.tableExists("truncate_demo")

### Where you'll use it

- **Refreshing a staging table:** `TRUNCATE` clears a load table quickly before reloading it; `DELETE` is used when only some rows must go and the change must be logged.

### Q: Why does `NOT IN` return no rows when the list contains NULL? Why does `COUNT(*)` differ from `COUNT(column)`?

**Answer.** SQL uses **three-valued logic**: a comparison with `NULL` is neither true nor false but unknown.
`x NOT IN (a, NULL)` means `x <> a AND x <> NULL`. The second part is unknown, so the whole condition is at best
unknown, and `WHERE` keeps only rows that are true. No row passes.

`COUNT(*)` counts rows; `COUNT(column)` counts the non-`NULL` values in that column. The NULL chapter covers both.

In [ ]:
q = "SELECT customer_id FROM customers WHERE customer_id NOT IN (SELECT * FROM VALUES (1), (NULL) AS t(id))"
assert spark.sql(q).collect() == []
assert spark.sql("SELECT 2 NOT IN (1, NULL)").collect() == [(None,)]

q = "SELECT COUNT(*), COUNT(x) FROM VALUES (1), (NULL), (3) AS t(x)"
assert spark.sql(q).collect() == [(3, 2)]

### Where you'll use it

- **Anti-joins in pipelines:** finding unmatched records with `NOT EXISTS` is the safe default, because one NULL in the list silently empties the result.

### Q: `UNION` vs `UNION ALL`?

**Answer.** Both stack the rows of two queries with the same columns. `UNION` then removes duplicate rows, which
costs a sort or hash step over the whole result; `UNION ALL` keeps every row. Stacking the customer ids of the orders
table with itself gives 28 rows with `UNION ALL` and 6 with `UNION`. The set operations chapter covers the rest.

In [ ]:
union_all = spark.sql("SELECT customer_id FROM orders UNION ALL SELECT customer_id FROM orders").count()
union = spark.sql("SELECT customer_id FROM orders UNION SELECT customer_id FROM orders").count()
assert (union_all, union) == (28, 6)

### Where you'll use it

- **Combining daily tables:** `UNION ALL` stacks daily extracts quickly; `UNION` is only needed when duplicates must be removed.

### Q: Where should a condition on the right-hand table of a left join go: `WHERE` or `ON`?

**Answer.** In `ON`. A condition in `WHERE` runs after the join. For a left row with no match, every right-hand column
is `NULL`, so `o.status = 'cancelled'` is unknown and the row is removed: the left join has become an inner join. A
condition in `ON` decides which right rows can match, and keeps every left row.

With "customers and their cancelled orders", the `ON` version keeps all 8 customers (two of them with a cancelled
order). The `WHERE` version keeps only customers 2 and 6.

In [ ]:
on_version = """
SELECT c.customer_id, o.order_id FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'cancelled'
"""
where_version = """
SELECT c.customer_id, o.order_id FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.status = 'cancelled'
"""
assert spark.sql(on_version).count() == 8
assert sorted(spark.sql(where_version).collect()) == [(2, 5), (6, 14)]

### Where you'll use it

- **Reports that must keep every customer:** a filter placed in `ON` keeps unmatched customers; the same filter in `WHERE` quietly turns the left join into an inner join.

## 5. More questions interviewers ask

Short questions that come up often, each with a small demo.

### Q: Why does `COUNT(*)` on an empty input return 0, while `MAX` returns NULL?

**Answer.** `COUNT` answers "how many", and the answer for no rows is 0. `SUM`, `AVG`, `MIN` and `MAX` summarise
values, and with no values there is nothing to report, so they return `NULL`. Wrap them in `COALESCE` when a report
needs zero.

In [ ]:
q = """
SELECT COUNT(*) AS count_of_nothing, MAX(amount) AS max_of_nothing, COALESCE(SUM(amount), 0) AS sum_or_zero
FROM orders WHERE 1 = 0
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(0, None, 0)]

### Where you'll use it

- **Empty-day reports:** a dashboard shows zero sales for a day with no orders, not a missing value, because `COUNT` returns 0 and `SUM` is wrapped in `COALESCE`.

### Q: What is data skew and how do you spot it?

**Answer.** **Skew** means a few keys hold most of the rows. In a join or a `GROUP BY`, all rows of one key go to one
task, so the task that handles the hot key does most of the work and everyone waits for it.

Spot it by counting rows per key and comparing the largest count with the average. In `orders` the largest customer
has 3 rows and the average is 14 / 6, about 2.33: a ratio near 1, so no skew. In the second example below one key holds
97 of 100 rows; the average over 4 keys is 25, so the ratio is 3.88 and one task would do almost all the work.

Fixes include broadcasting the small side of a join (the modeling chapter shows the `BROADCAST` hint), or **salting**
the hot key: adding a random suffix so its rows spread over several tasks, then combining the partial results.

In [ ]:
SKEW_RATIO = """
SELECT MAX(n) AS largest, ROUND(AVG(n), 2) AS average, ROUND(MAX(n) / AVG(n), 2) AS ratio
FROM (SELECT {key}, COUNT(*) AS n FROM {table} GROUP BY {key})
"""
q = SKEW_RATIO.format(key="customer_id", table="orders")
spark.sql(q).show()
assert spark.sql(q).collect() == [(3, 2.33, 1.29)]

spark.sql("""
CREATE OR REPLACE TEMP VIEW skewed AS
SELECT CASE WHEN id < 97 THEN 1 ELSE id - 95 END AS k FROM range(100)
""")
assert spark.sql(SKEW_RATIO.format(key="k", table="skewed")).collect() == [(97, 25.0, 3.88)]

### Where you'll use it

- **Slow joins at scale:** one customer or one key holds most rows, so one task runs for hours while others finish in seconds. Salting or a skew hint fixes it.

### Q: Why avoid `SELECT *` in production?

**Answer.** Columnar formats such as Parquet store each column separately, so the engine reads only the columns a
query names. `SELECT *` reads and transfers every column. It also changes shape when the table gains or reorders
columns, and it hides which columns a downstream consumer depends on.

The plan's `ReadSchema` lists the columns read from the files. Selecting `amount` reads one column; `SELECT *` reads
all five stored columns.

In [ ]:
def read_schema(query):
    return re.search(r"ReadSchema: (\S+)", plan(query)).group(1)


assert read_schema("SELECT amount FROM orders_files") == "struct<amount:int>"
assert read_schema("SELECT * FROM orders_files").count(":") == 5

### Where you'll use it

- **Cost and stability:** on a pay-per-scan warehouse, `SELECT *` reads every column and breaks downstream models when a column is added.
- **Code review:** the sargability and `SELECT *` points are the first comments on many pull requests.

### Q: Which filter is sargable: `WHERE YEAR(order_date) = 2023` or `WHERE order_date >= DATE '2023-01-01'`?

**Answer.** **Sargable** (from "search argument able") means the engine can use the condition to skip data, through
an index, partition pruning or file statistics. The second filter is sargable. It compares the bare column with a
constant, so the engine can check it against an index or against each file's stored minimum and maximum. The first
wraps the column in a function, so the engine must compute `YEAR` for every row before it can compare.

In Spark, the plan's `PushedFilters` shows which conditions are sent to the Parquet reader for skipping. The range
filter is pushed down; the `YEAR` filter is not. To keep the meaning of `YEAR(order_date) = 2023`, write the range
`order_date >= DATE '2023-01-01' AND order_date < DATE '2024-01-01'`.

In [ ]:
def pushed_filters(query):
    return re.search(r"PushedFilters: \[([^\]]*)\]", plan(query)).group(1)


assert "GreaterThanOrEqual(order_date" in pushed_filters(
    "SELECT amount FROM orders_files WHERE order_date >= DATE '2023-01-01'"
)
assert "order_date," not in pushed_filters("SELECT amount FROM orders_files WHERE YEAR(order_date) = 2023")

### Where you'll use it

- **Index and partition use:** `order_date >= '2023-01-01'` can use an index or skip files; `YEAR(order_date) = 2023` usually cannot.

### Q: How do you find the second-highest salary, or the top N per group?

**Answer.** For the second-highest, use `MAX` below the maximum, or `DENSE_RANK` equal to 2; both treat tied values
as one. For the top N per group, use `ROW_NUMBER` in a subquery, with a deterministic tie-breaker, and filter in the
outer query. The interview drills, Part A, work through both.

On `orders`, the distinct amounts start 500, 450, 400, so the second-highest is 450. The largest order per customer
is order 2 for customer 1, 6 for customer 2, 8 for customer 3, 10 for customer 4, 11 for customer 5 and 14 for customer 6.

In [ ]:
q = """
SELECT amount FROM (SELECT amount, DENSE_RANK() OVER (ORDER BY amount DESC) AS rnk FROM orders) WHERE rnk = 2
"""
assert spark.sql(q).collect() == [(450,)]

q = """
SELECT customer_id, order_id FROM (
  SELECT customer_id, order_id,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC, order_id) AS rn
  FROM orders
)
WHERE rn = 1
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 2), (2, 6), (3, 8), (4, 10), (5, 11), (6, 14)]

### Where you'll use it

- **Top-N per group:** the top three products in each category is a standard analytics question, solved with `DENSE_RANK` or `ROW_NUMBER` over a partition.

## Practice questions

1. Return the `order_id` of every order placed in February 2023, using a sargable filter on `orders`. Sort by `order_id`.
2. Write a query that returns the customers with no orders, using a method that is safe when the orders table can
   contain NULL customer ids. Show `customer_id`, sorted.
3. Return each customer whose completed revenue is above 500, with that revenue. Which clause filters the orders,
   and which filters the customers?
4. For every customer, count their cancelled orders, showing 0 for customers with none. Sort by `customer_id`.
5. Using the Type 2 dimension from section 2, which city does an order placed on 2023-05-31 join to?

## Answers

### Question 1

A half-open range on the bare column: on or after 1 February, and before 1 March. February's orders are 2, 5, 8, 9 and 13.

In [ ]:
q = """
SELECT order_id FROM orders
WHERE order_date >= DATE '2023-02-01' AND order_date < DATE '2023-03-01'
ORDER BY order_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2,), (5,), (8,), (9,), (13,)]

### Question 2

`NOT EXISTS` is safe with NULLs in the subquery, where `NOT IN` is not. Its answer is the same here because
`orders.customer_id` has no NULLs.

In [ ]:
q = """
SELECT c.customer_id
FROM customers c
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(7,), (8,)]

### Question 3

`WHERE` keeps only completed orders before grouping; `HAVING` keeps only the customers whose total exceeds 500.

In [ ]:
q = """
SELECT customer_id, SUM(amount) AS revenue
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
HAVING SUM(amount) > 500
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, 700), (5, 950)]

### Question 4

Put the status condition in `ON`, so customers without a cancelled order survive, and count a right-hand column so
their placeholder row counts as 0.

In [ ]:
q = """
SELECT c.customer_id, COUNT(o.order_id) AS cancelled
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'cancelled'
GROUP BY c.customer_id
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 0), (2, 1), (3, 0), (4, 0), (5, 0), (6, 1), (7, 0), (8, 0)]

### Question 5

Oslo. The Oslo row is valid from 2022-01-01 up to, but not including, 2023-06-01, and 2023-05-31 is inside that
interval. The Lima row starts the next day.

In [ ]:
q = """
SELECT d.city
FROM VALUES
  (1, 'Oslo', DATE '2022-01-01', DATE '2023-06-01'),
  (1, 'Lima', DATE '2023-06-01', DATE '9999-12-31')
AS d(customer_id, city, valid_from, valid_to)
WHERE DATE '2023-05-31' >= d.valid_from AND DATE '2023-05-31' < d.valid_to
"""
assert spark.sql(q).collect() == [("Oslo",)]